# Pokemon Type Guesser

## Imports & Dependencies
**-Data Handling & Math**
pandas,
numpy

**-Web Scraping & API**
requests,
beautifulsoup4

**-Natural Language Processing**
nltk,
transformers,
datasets

-**Machine Learning & Deep Learning**
torch,
scikit-learn

json, random, re, string, collections are already part of the **Python Standard Library**

We also download punkt_tab, punkt and stopwords from nltk

In [1]:
import requests
import json
import random
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re
import string
from bs4 import BeautifulSoup
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset, concatenate_datasets
import numpy as np
from collections import Counter
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import pandas as pd


nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('punkt')


[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Functions

This section is for functions I will be using in the future:

* **Stop Words** is using nltk to create a word list of mostly functional words that don't have important meanings in themselves.

* **Clean Text** makes the text lowercase, removes numbers, punctuation and special characters and returns the cleaned text

* **Tokenize Text** takes the string, tokenizes it and removes the instances of the pokemons name appearing, the word pokemon appearing and the alone s appearing after the removal of apostrophe. Afterwards it creates another list that doesn't have the stop words. This will help our model by removing the noice and only keeping the content words.

* **Preprocess Function** takes the prepared new dictionary makes it ready to be read by our LLM. It makes the list from **Tokenize Text** and turns it into a string and then tokenizes, word embeds and pads it as well as putting the pokemon abilities in the beginning so it will be easily processed by our LLM. It turns the pokemon types list into one hot encoding since I will be using DistilBert an encoding-only LLM.

* **Compute Metrics** is making sure I have the performance score for **F1 Macro** score which prioritizes generally good f1 scores for each type and **F1 Micro** score which prioritizes generally good f1 score in total.

* **Show Predictions** takes some elements from validation set and sees how successful the model is at predicting them.

* **Independent Test** takes new description and type combinations and sees how successful the model is at predicting them.

In [2]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower() 
    text = re.sub(r'\d+', '', text) 
    text = text.translate(str.maketrans('', '', string.punctuation)) 
    text = re.sub(r'\W', ' ', text) 
    return text
    
def tokenize_text(text):
    tok_desc = nltk.word_tokenize(text)
    while name.lower() in tok_desc:
      tok_desc.remove(name.lower())
    for f in tok_desc:
      if 'pokémon' in f:
        tok_desc.remove(f)
    while 's' in tok_desc:
      tok_desc.remove('s')
    filt_desc = [word for word in tok_desc if word not in stop_words]
    return filt_desc

def preprocess_function(examples):
    combined_texts = []
    for ability_list, desc_list in zip(examples["abilities"], examples["description"]):
        text= " ".join(ability_list) + " " + " ".join(desc_list)
        combined_texts.append(text.lower())
    tokenized = tokenizer(
        combined_texts,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    num_categories = len(types) 

    batch_labels = []
    for itypes in examples["actual_types"]:
        one_hot = np.zeros(num_categories, dtype=np.float32)

        for t in itypes:
            if t in label2id:
                idx = label2id[t]
                one_hot[idx] = 1.0

        batch_labels.append(one_hot)

    tokenized["labels"] = batch_labels
    return tokenized

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.Tensor(logits)).numpy()

    predictions = (probs >= 0.2).astype(float)

    f1_micro = f1_score(labels, predictions, average='micro')
    f1_macro = f1_score(labels, predictions, average='macro')
    accuracy = accuracy_score(labels, predictions)

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "accuracy": accuracy
    }

def show_predictions(num_examples=3):
    model.eval()
    indices = random.sample(range(len(val_set)), num_examples)
    correct = 0
    miss = 0
    for idx in indices:
        item = val_set[idx]
        
        input_ids = torch.tensor(item['input_ids']).unsqueeze(0).to(model.device)
        attention_mask = torch.tensor(item['attention_mask']).unsqueeze(0).to(model.device)

        with torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs = torch.sigmoid(logits).cpu().numpy()[0]

        predicted_types = [id2label[i] for i, p in enumerate(probs) if p >= 0.3]
        actual_types = [id2label[i] for i, val in enumerate(item['labels']) if val == 1.0]

        raw_text = tokenizer.decode(item['input_ids'], skip_special_tokens=True)

        correct_hits = set(predicted_types) & set(actual_types)
        if correct_hits:
            correct += 1
        else:
            miss += 1
    print(f"Guessed full or partially correct: {correct} times")
    print(f"Guessed wrong: {miss} times")
def independent_test(descriptions, actual_types_list, threshold=0.5):
    print("--- Independent Evaluation ---")
    answer = 0
    wrong = 0
    for i, text in enumerate(descriptions):
        clean_desc = clean_text(text)
        final_desc = " ".join(tokenize_text(clean_desc))
        
        inputs = tokenizer(final_desc, return_tensors="pt", truncation=True, padding=True).to(model.device)

        with torch.no_grad():
            logits = model(**inputs).logits

        probs = torch.sigmoid(logits).cpu().numpy()[0]

        predicted = [id2label[idx] for idx, p in enumerate(probs) if p >= threshold]
        actual = actual_types_list[i]

        if len(set(predicted).intersection(set(actual))) != 0:
            answer += 1
        else:
            wrong += 1
    print(f"Guessed full or partially correct: {answer} times")
    print(f"Guessed wrong: {wrong} times")

## Getting and Processing the Data

Here I get the data from its github url, take only name, abilities, description and types from the whole dictionary per pokemon. Then I preprocess them, add them to a list and shuffle their order.

In [3]:
url = "https://raw.githubusercontent.com/Purukitto/pokemon-data.json/master/pokedex.json"
response = requests.get(url)
data = response.json()

processed_data = []
for p in data:
    name = p['name']['english']
    pok_name = f"{name}"
    
    open_abilities = [nltk.word_tokenize(a[0]) for a in p['profile']['ability']]
    abilities = [item for sublist in open_abilities for item in sublist]
    
    desc = p['description']
    clean_desc = clean_text(desc)
    final_desc = tokenize_text(clean_desc)
    actual_types = p['type']

    processed_data.append({
        "name": [pok_name],
        "abilities": abilities,
        "description": final_desc,
        "actual_types": actual_types
    })

random.seed(42) 
random.shuffle(processed_data)
train_set = processed_data

## Saving the Types

Since sometimes pokemon adds new types with new games and I am not sure to which generation this data goes to, instead of making a list of all pokemon types, I make a list from the types there are in the data.

In [4]:
types= []

for t in processed_data:
  for y in t['actual_types']:
    if y not in types:
      types.append(y)

label2id = {t: i for i, t in enumerate(types)}
id2label = {i: t for i, t in enumerate(types)}

## Loading the Model

I use DistilBert for its relatively small size and its usefulness for Multi Label Classification. I will fine-tune it for guessing pokemon types next.

In [5]:
model_id = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=len(types),
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Setting Everything Up for the Training

I will use F1 Micro for the training metric. If I were to see a low F1 Macro score I would've considered changing it but it was pretty consistent with the Micro scores. The number of epochs and the learning rate are both things I ended up with after several experiments with other numbers.

In [6]:
dataset = Dataset.from_list(train_set).map(preprocess_function, batched=True)
training_args = TrainingArguments(
    output_dir="./lightweight_pokemon_classifier",
    eval_strategy="epoch",        
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    report_to="none"
)

if "test" not in dataset:
    split_data = dataset.train_test_split(test_size=0.2)
    train_set = split_data["train"]
    val_set = split_data["test"]
else:
    train_set = dataset["train"]
    val_set = dataset["test"]



Map:   0%|          | 0/898 [00:00<?, ? examples/s]

## Dummy Pokemons

Since there is no way for our model to see pokemon types as their semantic meaning instead of a 18 number array, I will put a bunch of dummy pokemons with descriptions just made up of its type. This will help the model associate the array positions with their semantic contents.

In [7]:
makala = []
for i in types:
  for n in range(10):
    u = i.lower()
    makala.append({
          "name": [f"Dummy {i}"],
          "abilities": [u,u],
          "description": [u,u,u,u,u,u,u,u,u,u],
          "actual_types": [i]
      })
datasetmak = Dataset.from_list(makala).map(preprocess_function, batched=True)
result = concatenate_datasets([train_set, datasetmak])

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

## Training

This is just training. It will print Training Loss, Validation Loss, F1 Micro, F1 Macro and Accuracy every epoch.

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=result,
    eval_dataset=val_set,          
    compute_metrics=compute_metrics,
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy
1,0.672207,0.552039,0.000000,0.000000,0.000000
2,0.535927,0.519751,0.288889,0.168153,0.177778
3,0.427137,0.406209,0.550832,0.498034,0.311111
4,0.289552,0.344369,0.612717,0.581611,0.438889
5,0.197373,0.331312,0.614525,0.584608,0.400000
6,0.140052,0.326247,0.640625,0.611274,0.461111
7,0.105971,0.325820,0.623853,0.595504,0.450000
8,0.086415,0.321905,0.627523,0.603592,0.455556


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=232, training_loss=0.30682926136871863, metrics={'train_runtime': 61.5436, 'train_samples_per_second': 116.73, 'train_steps_per_second': 3.77, 'total_flos': 237979332993024.0, 'train_loss': 0.30682926136871863, 'epoch': 8.0})

## Evaluation

After the automatic evaluation I decided to evaluate it with new entries. Unseen texts are ai generated or custom made by me and my friends, while Gen9 texts are from the descriptions of Gen9 pokemons since I definitely know they are not in the github database.

In [9]:
model.eval()

unseen_texts = [
    "A magnificent bird whose plumage blazes with eternal fire, lighting up the night sky.", 
    "A giant flower that emits a foul-smelling, highly toxic powder to paralyze its prey.",
    "It swims through freezing oceans, using its heavy tusks to shatter glaciers.",
    "It lurks in the deepest shadows, feeding on the nightmares and fears of lost travelers.",
    "Encased in an impenetrable metallic shell, this insect fires razor-sharp spikes from its wings.",
    "A colossal serpent made of jagged boulders that burrows effortlessly through the earth's crust.",
    "A mystical guardian of the forest that uses telekinesis to levitate and soothe enraged beasts.",
    "A fluffy, unremarkable mammal that runs away and hides at the first sign of danger.",
    "It oozes a purple, acidic sludge that rapidly melts anything it touches.",
    "A robotic canine with a battery for a heart, discharging high-voltage electricity when it barks.",
    "A hot-headed brawler whose fists ignite with blazing heat during intense hand-to-hand combat.",
    "It is said to be born from an icicle. Its breath is so cold it freezes moisture in the air instantly.",
    "A gentle giant of the sea that swallows massive amounts of ocean currents and blasts them out like a geyser.",
    "It resembles a tiny mushroom. It dances in moonlight, spreading sparkling spores that bring sweet dreams.",
    "A serpentine beast of the sky that summons raging hurricanes and thunderstorms with a single flap of its wings.",
    "A treacherous desert dweller that creates quicksand traps to drag unsuspecting victims into the abyss.",
    "It buzzes rapidly, darting through the air to drain sap from trees with its needle-like mouth.",
    "An ancient creature revived from a fossil. It uses its heavy, stone-like shell to crush prey on the ocean floor.",
    "An abandoned sword possessed by a vengeful spirit, it drains the life force of anyone who grasps its metallic hilt.",
    "It sits in deep meditation for years, developing an intelligence so vast it can foresee the future.",
    "A small, golden-furred mouse with indigo eyes and cowl-like ears that trap morning dew. Legend says it can see the paths of the wind before they blow, and those who follow its tracks across the deep dunes will never lose their way.",
    " loves music and high places, and you’ll often see it beating out a rhythm or dancing. When it uses its stick to strike up a beat, even the plants and flowers in the area seem happier and revitalized.",
    "It skillfully uses its two sticks not only for drumming out rapid beats but also for pummeling opponents during battle.",
    "Though it has a calm disposition, it won’t tolerate those who drum up discord. It strictly disciplines offenders until they learn their lesson.",
    "Once its body has heated up, can use the full extent of its power. That’s why it does warm-up exercises.",
    "Thanks to its soft and fluffy fur, can easily heat up its fire energy and produce flames with more power.",
    "Although  loses its cool easily, it will battle flawlessly for it trusts.",
    "It gets berries to eat by shooting them down with bullets of water it spurts from its mouth. Its aim is perfect.",
    "It extends its long, slimy tongue at blinding speeds and finishes off its prey with great skill.",
    "It uses special lenses in its eyes to sense things about its opponents—like their body heat—then attacks once it identifies a weak spot."

]

unseen_types = [
    ["Fire", "Flying"],
    ["Grass", "Poison"],
    ["Water", "Ice"],
    ["Ghost", "Dark"],
    ["Bug", "Steel"],
    ["Rock", "Ground"],
    ["Fairy", "Psychic"],
    ["Normal"],
    ["Poison"],
    ["Electric", "Steel"],
    ["Fighting", "Fire"],
    ["Ice"],
    ["Water"],
    ["Grass", "Fairy"],
    ["Dragon", "Flying"],
    ["Ground", "Dark"],
    ["Bug", "Flying"],
    ["Rock", "Water"],
    ["Ghost", "Steel"],
    ["Psychic"],
    ['Ground', 'Psychic'],
    ['Grass'],
    ['Grass'],
    ['Grass'],
    ['Fire'],
    ['Fire'],
    ['Fire'],
    ['Water'],
    ['Water'],
    ['Water']
]

gen9_texts = [
    # 1. Sprigatito
    "A capricious feline that kneads with its front paws to unleash a sweet, relaxing aroma.",
    # 2. Fuecoco
    "It lies on warm rocks and uses the heat absorbed by its square scales to create fire energy.",
    # 3. Quaxly
    "An earnest duck-like bird with coif-like feathers on its head that secrete a glossy gel to repel water and grime.",
    # 4. Tinkaton
    "This intelligent but aggressive creature swings a massive, custom-built hammer to knock rocks into the sky.",
    # 5. Clodsire
    "It lives at the bottom of ponds and swamps, retaliating against attackers by sticking thick, poisonous spines out from its body.",
    # 6. Ceruledge
    "Clad in armor steeped in grudges, it wields blades made of fire and phantom energy that leave wounds which burn life force.",
    # 7. Armarouge
    "It wears armor that belonged to a distinguished warrior, using the mental power within it to control intense fire.",
    # 8. Kilowattrel
    "It inflates its throat sac to amplify its electricity, riding the wind to cover huge distances while crackling with power.",
    # 9. Mabosstiff
    "Though it has a gentle disposition, it can easily intimidate opponents with its massive fangs and menacing, shadowy glare.",
    # 10. Houndstone
    "A skeletal canine resting in graveyards. It has a tombstone on its head and drains the life force of anyone it plays with.",
    # 11. Palafin
    "It looks like a normal dolphin, but when its allies are in danger, it transforms into a muscular hero of the seas.",
    # 12. Baxcalibur
    "It blasts cryogenic air from its mouth that can freeze even hot magma, and uses its dorsal blade to finish off enemies.",
    # 13. Gholdengo
    "Its body is composed of exactly 1,000 gold coins. It surfs on a golden board and can manipulate its metallic body shape freely.",
    # 14. Meowscarada
    "A bipedal feline magician that uses sleight of hand to plant explosive flower bombs on its unsuspecting foes.",
    # 15. Skeledirge
    "It sings a mournful song while a fiery bird rests on its snout, born from the life energy escaping its body.",
    # 16. Quaquaval
    "A passionate dancer that uses its powerful legs to perform elegant, fluid kicks that can slice through bedrock.",
    # 17. Garganacl
    "A colossal golem made of pure salt. It rubs its fingertips together to sprinkle salt that quickly heals wounds.",
    # 18. Annihilape
    "Its anger grew so intense that it transcended its physical body, gaining immense power unbound by mortal limits.",
    # 19. Iron Valiant
    "A futuristic, mechanical warrior created by a mad scientist, wielding a double-sided laser halberd.",
    # 20. Slither Wing
    "An ancient, prehistoric insect that aggressively charges at foes, using its massive physical strength and fuzzy body to brawl."
]

gen9_types = [
    ["Grass"],                       # 1. Sprigatito
    ["Fire"],                        # 2. Fuecoco
    ["Water"],                       # 3. Quaxly
    ["Fairy", "Steel"],              # 4. Tinkaton
    ["Poison", "Ground"],            # 5. Clodsire
    ["Fire", "Ghost"],               # 6. Ceruledge
    ["Fire", "Psychic"],             # 7. Armarouge
    ["Electric", "Flying"],          # 8. Kilowattrel
    ["Dark"],                        # 9. Mabosstiff
    ["Ghost"],                       # 10. Houndstone
    ["Water"],                       # 11. Palafin
    ["Dragon", "Ice"],               # 12. Baxcalibur
    ["Steel", "Ghost"],              # 13. Gholdengo
    ["Grass", "Dark"],               # 14. Meowscarada
    ["Fire", "Ghost"],               # 15. Skeledirge
    ["Water", "Fighting"],           # 16. Quaquaval
    ["Rock"],                        # 17. Garganacl
    ["Fighting", "Ghost"],           # 18. Annihilape
    ["Fairy", "Fighting"],           # 19. Iron Valiant
    ["Bug", "Fighting"]              # 20. Slither Wing
]

show_predictions(10)
independent_test(unseen_texts, unseen_types, threshold=0.4)
independent_test(gen9_texts, gen9_types, threshold=0.4)

Guessed full or partially correct: 9 times
Guessed wrong: 1 times
--- Independent Evaluation ---
Guessed full or partially correct: 15 times
Guessed wrong: 15 times
--- Independent Evaluation ---
Guessed full or partially correct: 5 times
Guessed wrong: 15 times
